In [1]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from openmmtools.integrators import LangevinIntegrator, LangevinSplittingGirsanov
from openmmplumed import PlumedForce
import matplotlib

from deeptime.decomposition import TICA

import numpy as np
import MDAnalysis as md
from matplotlib import pyplot as plt

from deeptime.clustering import MiniBatchKMeans
from deeptime.markov import TransitionCountEstimator
from deeptime.markov.msm import MaximumLikelihoodMSM
from deeptime.plots import plot_implied_timescales
from deeptime.util.validation import implied_timescales
from deeptime.markov import GirsanovReweightingEstimator

from scipy.linalg import eig
from scipy.stats import gaussian_kde

from pymbar import MBAR
import plumed
import subprocess

from scipy.interpolate import CubicSpline

import gc

import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['font.size'] = 20

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************

/home/mingyuan/miniconda3/envs/girsanov-torch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readt

In [2]:
def build_MSM(msm_lagtime,assignments):
    counts = TransitionCountEstimator(lagtime=msm_lagtime, count_mode='sliding').fit_fetch(assignments)
    msm = MaximumLikelihoodMSM().fit_fetch(counts)
    return msm

def Girsanov_MSM(msm_lagtime,assignments,reweighting_factors,stationary_pi=None):
    count_estimator = GirsanovReweightingEstimator(lagtime=msm_lagtime,count_mode='sliding')
    counts = count_estimator.fit(data=assignments,reweighting_factors=reweighting_factors).fetch_model()
    if stationary_pi is not None:
        msm = MaximumLikelihoodMSM(stationary_distribution_constraint=stationary_pi).fit_fetch(counts)
    else:
        msm = MaximumLikelihoodMSM().fit_fetch(counts)
    return msm

# We need to further modify assignments_unbiased and assignments_biased
def reorganize_indices(assignments,n_microstates):
    populated_states = np.unique(assignments)
    empty_states = []
    assignments_new = np.zeros(assignments.shape,dtype=np.int32)
    shift = 0
    for idx in range(n_microstates):
        if idx in populated_states:
            idx_new = idx - shift
            assignments_new[np.where(assignments == idx)] = idx_new
        else:
            empty_states.append(idx)
            shift += 1
    return assignments_new, empty_states

def reorganize_eigenvectors(msm,n_eigenvectors,n_microstates,empty_states,align=None):
    eigenvectors_new = np.zeros((n_microstates,n_eigenvectors))
    shift = 0
    for idx in range(n_microstates):
        if idx in empty_states:
            shift += 1
            empty_entries = np.zeros(n_eigenvectors,)
            # Let values of eigenvectors being nan for plotting
            empty_entries[:] = np.nan
            eigenvectors_new[idx,:] = empty_entries
        else:
            idx_new = idx - shift
            pi_idx = msm.stationary_distribution[idx_new]
            ev_idx = msm.eigenvectors_right()[idx_new,1:n_eigenvectors]
            eigenvectors_new[idx,:] = np.concatenate([[pi_idx],ev_idx])
    if align is not None:
        # Align each eigenvector with the reference eigenvector
        for i in range(1,n_eigenvectors):
            if np.nansum(np.abs(-eigenvectors_new[:,i] - align[:,i])) < np.nansum(np.abs(eigenvectors_new[:,i] - align[:,i])):
                eigenvectors_new[:,i] = -eigenvectors_new[:,i]
    return eigenvectors_new

### 0. Implicit Solvent Ala2 simulation with OpenMM

#### 0.1 System Setup

In [57]:
# Parameters
pdb_file = 'traj_and_dat/ala2/input.pdb'
protein_ff = 'amber99sbildn.xml'
solvent_ff = 'amber99_obc.xml'      # obc is implicit solvent model
temp = 310
temperature = temp * kelvin
friction = 100 / picosecond
timestep = 2 * femtosecond
splitting = 'R V O V R'
nstxout = 50
kt = temp*8.314/1000
platform = Platform.getPlatformByName('CUDA')

#### 0.2 Unbiased Ref Simulation

In [59]:
simulation_length = 500000000          # 1us
reporting_frequency = 500000           # 1ns
n_simulations = 1                      # Run 5 trajectories

for i in range(n_simulations):
    pdb = PDBFile(pdb_file)
    forcefield = ForceField(protein_ff,solvent_ff)
    system = forcefield.createSystem(pdb.topology,constraints=HBonds)
    integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                           timestep=timestep,splitting=splitting)
    
    simulation = Simulation(pdb.topology,system,integrator)
    simulation.context.setPositions(pdb.positions)
    simulation.minimizeEnergy()
    simulation.reporters.append(DCDReporter('traj_and_dat/ala2/ala2-ref.dcd'.format(i=i),nstxout))
    simulation.reporters.append(StateDataReporter(
        stdout,reporting_frequency,step=True,remainingTime=True,
        speed=True,totalSteps=simulation_length))
    simulation.step(simulation_length)

#"Step","Speed (ns/day)","Time Remaining"
500000,0,--
1000000,1.7e+03,14:07:04
1500000,1.69e+03,14:11:48
2000000,1.68e+03,14:15:21
2500000,1.67e+03,14:17:26
3000000,1.67e+03,14:19:36


KeyboardInterrupt: 

### 1. Phi, Psi as 1D reaction coordinate

#### 1.1 Build-up Only (310K)
Obtain stationary distribution with the assumption of pseudostatic bias.

In [ ]:
# Metadynamics parameters
biasfactor = 1.25
cv_sigma = 0.1
periodic_cv = True
gridwidth = 100
height = 1.2 * kilojoule_per_mole
pace = 500

In [452]:
# Phi Build-up Simulation
name = 'phi_build_up'
simulation_length = 50000000              # 100ns
n_iters = int(simulation_length / nstxout)       
reporting_frequency = 50000               # 10ps

M = np.zeros(n_iters,)

topology = PDBFile(pdb_file).topology
forcefield = ForceField(protein_ff,solvent_ff)

system = forcefield.createSystem(topology,constraints=HBonds)

# Define the collective variable as the phi angle, C-N-CA-C
# (idx in pdb 5,7,9,15, remember -1 since openmm starts with 0)

cv_force = CustomTorsionForce("theta")
cv_force.addTorsion(4,6,8,14,[])

# Create the BiasVariable and Metadynamics objects
cv = BiasVariable(
    cv_force, -np.pi, np.pi, cv_sigma, periodic_cv, gridwidth
)
    
metad = Metadynamics(
    system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
)

# Set ForceGroup to 1 for Girsanov path weights
metad._force.setForceGroup(1)

integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                       timestep=timestep,splitting=splitting)

simulation = Simulation(pdb.topology,system,integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.reporters.append(DCDReporter('traj_and_dat/ala2/{name}.dcd'.format(name=name),nstxout))
simulation.reporters.append(StateDataReporter(
    stdout,reporting_frequency,step=True,remainingTime=True,
    speed=True,totalSteps=simulation_length))

for k in range(n_iters):
    metad.step(simulation,nstxout)
    M[k] = simulation.integrator.getGlobalVariableByName("M")

np.save('traj_and_dat/ala2/M_{name}.npy'.format(name=name), M)
np.save('traj_and_dat/ala2/Bias_{name}.npy'.format(name=name), metad._totalBias)

#"Step","Speed (ns/day)","Time Remaining"
50000,0,--
100000,1.17e+03,2:02:32
150000,1.17e+03,2:02:43
200000,1.16e+03,2:03:19
250000,1.16e+03,2:03:20
300000,1.16e+03,2:03:20
350000,1.16e+03,2:03:03
400000,1.17e+03,2:02:30
450000,1.17e+03,2:02:21
500000,1.17e+03,2:02:00
550000,1.17e+03,2:01:48
600000,1.17e+03,2:01:49
650000,1.17e+03,2:01:35
700000,1.17e+03,2:01:36
750000,1.17e+03,2:01:24
800000,1.17e+03,2:01:14
850000,1.17e+03,2:01:11
900000,1.17e+03,2:00:58
950000,1.17e+03,2:00:52
1000000,1.17e+03,2:00:47
1050000,1.17e+03,2:00:42
1100000,1.17e+03,2:00:37
1150000,1.17e+03,2:00:27
1200000,1.17e+03,2:00:16
1250000,1.17e+03,2:00:07
1300000,1.17e+03,1:59:55
1350000,1.17e+03,1:59:44
1400000,1.17e+03,1:59:33
1450000,1.17e+03,1:59:24
1500000,1.17e+03,1:59:17
1550000,1.17e+03,1:59:06
1600000,1.17e+03,1:58:57
1650000,1.17e+03,1:58:47
1700000,1.17e+03,1:58:38
1750000,1.17e+03,1:58:28
1800000,1.17e+03,1:58:20
1850000,1.17e+03,1:58:11
1900000,1.17e+03,1:58:01
1950000,1.17e+03,1:57:53
2000000,1.17e+0

In [453]:
# Psi Build-up Simulation
name = 'psi_build_up'
simulation_length = 25000000              # 50ns
n_iters = int(simulation_length / nstxout)       
reporting_frequency = 50000               # 10ps

M = np.zeros(n_iters,)

topology = PDBFile(pdb_file).topology
forcefield = ForceField(protein_ff,solvent_ff)

system = forcefield.createSystem(topology,constraints=HBonds)

# Define the collective variable as the phi angle, N-CA-C-N
# (idx in pdb 7,9,15,17 remember -1 since openmm starts with 0)

cv_force = CustomTorsionForce("theta")
cv_force.addTorsion(6,8,14,16,[])

# Create the BiasVariable and Metadynamics objects
cv = BiasVariable(
    cv_force, -np.pi, np.pi, cv_sigma, periodic_cv, gridwidth
)
    
metad = Metadynamics(
    system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
)

# Set ForceGroup to 1 for Girsanov path weights
metad._force.setForceGroup(1)

integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                       timestep=timestep,splitting=splitting)

simulation = Simulation(pdb.topology,system,integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.reporters.append(DCDReporter('traj_and_dat/ala2/{name}.dcd'.format(name=name),nstxout))
simulation.reporters.append(StateDataReporter(
    stdout,reporting_frequency,step=True,remainingTime=True,
    speed=True,totalSteps=simulation_length))

for k in range(n_iters):
    metad.step(simulation,nstxout)
    M[k] = simulation.integrator.getGlobalVariableByName("M")

np.save('traj_and_dat/ala2/M_{name}.npy'.format(name=name), M)
np.save('traj_and_dat/ala2/Bias_{name}.npy'.format(name=name), metad._totalBias)

#"Step","Speed (ns/day)","Time Remaining"
50000,0,--
100000,272,4:23:18
150000,287,4:09:47
200000,282,4:13:42
250000,279,4:15:44
300000,277,4:16:32
350000,276,4:16:46
400000,285,4:08:20
450000,284,4:09:18
500000,282,4:10:03
550000,279,4:12:16
600000,278,4:12:30
650000,278,4:12:37
700000,277,4:12:37
750000,275,4:14:05
800000,275,4:13:47
850000,274,4:13:30
900000,274,4:13:14
950000,277,4:10:24
1000000,276,4:10:04
1050000,276,4:09:51
1100000,276,4:09:30
1150000,276,4:09:08
1200000,276,4:08:44
1250000,275,4:08:23
1300000,275,4:08:02
1350000,275,4:07:39
1400000,275,4:07:17
1450000,275,4:06:51
1500000,275,4:06:24
1550000,275,4:05:56
1600000,275,4:05:29
1650000,274,4:05:04
1700000,274,4:04:35
1750000,274,4:04:07
1800000,274,4:03:50
1850000,274,4:03:22
1900000,274,4:02:53
1950000,274,4:02:24
2000000,274,4:01:54
2050000,274,4:01:24
2100000,273,4:01:09
2150000,273,4:00:39
2200000,273,4:00:09
2250000,273,3:59:44
2300000,274,3:58:44
2350000,275,3:57:27
2400000,275,3:56:58
2450000,275,3:56:30
25000